# Course: 65007 - NLP and Speech Analysis
 **Program:** Intelligent Systems  
 **Course coordinator:** Dr. Sharon Yalov-Handzel

**Submission for:**  Assignment 2   
**by**  
  - Michael Berger, 318063864  
  - Barack Samuni, 318299625

# 1. Word2Vec

## a. Write Python program to implement Skip-gram Word2Vec algorithm.
Barak

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.tokenize import word_tokenize

# Download necessary NLTK resources
nltk.download('punkt')


class SkipGramDataset(Dataset):
    def __init__(self, tokenized_corpus, word_to_index, window_size):
        """
        Custom dataset for Skip-gram Word2Vec.
        Generates target-context word pairs based on the pre-tokenized corpus.
        """
        self.pairs = []  # Store target-context pairs

        # Generate target-context pairs
        for sentence in tokenized_corpus:
            sentence_indices = [word_to_index[word] for word in sentence]
            for i, target_index in enumerate(sentence_indices):
                # Create a context window around the target word
                context_indices = sentence_indices[max(0, i - window_size):i] + \
                                  sentence_indices[i + 1:min(len(sentence_indices), i + window_size + 1)]
                for context_index in context_indices:
                    # Append the (target, context) pair to the list
                    self.pairs.append((target_index, context_index))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        return self.pairs[index]


class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, embedding_size):
        """
        Skip-gram Word2Vec model with trainable embeddings.
        """
        super(SkipGramModel, self).__init__()
        self.input_embeddings = nn.Embedding(vocab_size, embedding_size)    # target words to dense embedding vectors
        self.output_embeddings = nn.Embedding(vocab_size, embedding_size)   # context words to dense embedding vectors

    def forward(self, target_words, context_words):
        """
        Forward pass to compute the logits (scores) for target-context word pairs.
        """
        # Embedding lookup for target and context words
        target_embeds = self.input_embeddings(target_words)     # Shape: (batch_size, embedding_size)
        context_embeds = self.output_embeddings(context_words)  # Shape: (batch_size, embedding_size)

        # Compute dot product (logits) between target and context embeddings
        logits = torch.sum(target_embeds * context_embeds, dim=1)  # Shape: (batch_size)

        return logits

def train_skipgram_model(corpus, embedding_size=10, window_size=2, learning_rate=0.01, epochs=10, batch_size=64):
    """
    Trains a Skip-gram Word2Vec model with PyTorch. Handles tokenization and vocab generation internally.
    """
    # Tokenization and Vocabulary Creation
    tokenized_corpus = [word_tokenize(sentence.lower()) for sentence in corpus]
    words = [word for sentence in tokenized_corpus for word in sentence]
    vocab = list(set(words))
    word_to_index = {word: i for i, word in enumerate(vocab)}
    index_to_word = {i: word for word, i in word_to_index.items()}
    vocab_size = len(word_to_index)

    # Create Skip-gram dataset and dataloader
    dataset = SkipGramDataset(tokenized_corpus, word_to_index, window_size)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Initialize the SkipGram model
    model = SkipGramModel(vocab_size, embedding_size)
    criterion = nn.CrossEntropyLoss()  # Negative log likelihood loss with softmax
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Get the training device (CPU or GPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training device: {'GPU (CUDA)' if torch.cuda.is_available() else 'CPU'}")
    model.to(device)

    # Training loop
    for epoch in range(epochs):
        total_loss = 0
        for target, context in dataloader:
            # Send tensors to the training device
            target = target.to(device)
            context = context.to(device)

            # Forward pass
            logits = model.input_embeddings(target) @ model.output_embeddings.weight.T  # Shape: (batch_size, vocab_size)

            # Compute loss
            loss = criterion(logits, context)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss:.4f}")

    # Extract word embeddings to a dictionary
    embeddings = model.input_embeddings.weight.cpu().detach().numpy()
    embedding_dict = {index_to_word[i]: embeddings[i] for i in range(vocab_size)}

    return embedding_dict

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\barak\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## b. Write Python program that implements CBOW Word2Vec algorithm
Michael

## c. Apply both programs to the following text:

```
i. The bank is located near the river.
ii. The bank approved my loan application.
iii. He rose from his chair to close the window.
iv. The rose bloomed beautifully in the garden.
v. The lead actor delivered a stunning performance.
vi. Exposure to lead is harmful to health.
vii. She is reading a book in the library.
viii. The book mentioned a fascinating historical event.
ix. I need to file a report for my manager.
x. He lost the file containing important documents.
```

Skip-gram

In [10]:
corpus = [
    "The bank is located near the river.",
    "The bank approved my loan application.",
    "He rose from his chair to close the window.",
    "The rose bloomed beautifully in the garden.",
    "The lead actor delivered a stunning performance.",
    "Exposure to lead is harmful to health.",
    "She is reading a book in the library.",
    "The book mentioned a fascinating historical event.",
    "I need to file a report for my manager.",
    "He lost the file containing important documents."
]
embedding_dict_skipgram = train_skipgram_model(corpus)
embedding_dict_skipgram

Training device: GPU (CUDA)
Epoch 1/10, Loss: 36.8129
Epoch 2/10, Loss: 34.0520
Epoch 3/10, Loss: 32.1559
Epoch 4/10, Loss: 30.3851
Epoch 5/10, Loss: 29.1702
Epoch 6/10, Loss: 27.4911
Epoch 7/10, Loss: 26.6007
Epoch 8/10, Loss: 25.3143
Epoch 9/10, Loss: 24.2665
Epoch 10/10, Loss: 23.6379


{'he': array([-1.2009562 ,  0.09446239, -0.62522995,  0.05828322, -0.26441813,
         0.26531664,  0.390266  , -0.3942018 ,  0.15618093,  0.26379827],
       dtype=float32),
 'mentioned': array([-0.18618171,  0.39249304,  1.3484132 ,  0.06792637, -0.9293208 ,
         1.5880983 ,  1.0234445 ,  1.1302633 ,  0.8156896 , -1.5344201 ],
       dtype=float32),
 'reading': array([ 1.1067728 , -0.2366533 , -1.2428963 ,  0.1637473 , -0.6696501 ,
        -0.30681115, -0.79040825, -0.8601573 ,  0.44870508,  0.00859739],
       dtype=float32),
 'a': array([-0.43903244, -1.7169126 , -1.1258068 ,  0.45893404, -1.0679713 ,
        -0.46620524,  0.5493672 ,  0.3674028 ,  0.9452898 ,  0.64792705],
       dtype=float32),
 'is': array([ 0.5592894 , -0.05521967, -0.5305127 ,  0.08293814,  0.83313644,
        -2.0258982 , -1.1458248 ,  0.29426628, -0.542919  , -0.0677876 ],
       dtype=float32),
 'she': array([ 0.01218892,  1.2505721 , -0.78648984,  0.91646934,  0.5630935 ,
        -0.7146004 , -2.11086

CBOW

## d. What is the difference between the embeddings? 
Explain the results.

## e. Can you find a text that 
Its embedding will be similar in these two algorithms?

## f. Repeat step c with different window sizes. 
Is there a significant change?

### Skip-gram

In [14]:
import pandas as pd  # Import pandas to handle the DataFrame

window_sizes = [1, 2, 3, 4, 5]

# To hold embeddings for comparison and results
previous_embedding_dict = None
differences_data = []  # To store differences

for window_size in window_sizes:
    embedding_dict = train_skipgram_model(corpus, window_size=window_size)
    print(f"Window size: {window_size}\n")
    print("----------------------------------------\n")

    if previous_embedding_dict is not None:
        row_data = {}  # Row data for the DataFrame

        for word in embedding_dict:
            if word in previous_embedding_dict:
                difference = embedding_dict[word] - previous_embedding_dict[word]
                row_data[word] = difference  # Store difference
            else:
                print(f"Word: {word} is new in the current embedding.\n")

        differences_data.append(row_data)  # Append the differences for this window size comparison
        print("----------------------------------------\n")

    previous_embedding_dict = embedding_dict  # Update for the next comparison

# Create a DataFrame from the differences data
comparison_labels = [f"{window_sizes[i]}-{window_sizes[i+1]}" for i in range(len(window_sizes)-1)]
differences_df = pd.DataFrame(differences_data, index=comparison_labels)
print("Differences DataFrame:")
differences_df

Training device: GPU (CUDA)
Epoch 1/10, Loss: 23.6943
Epoch 2/10, Loss: 23.0757
Epoch 3/10, Loss: 22.5544
Epoch 4/10, Loss: 21.8034
Epoch 5/10, Loss: 20.3383
Epoch 6/10, Loss: 20.1553
Epoch 7/10, Loss: 18.3800
Epoch 8/10, Loss: 18.1594
Epoch 9/10, Loss: 18.5135
Epoch 10/10, Loss: 17.1994
Window size: 1

----------------------------------------

Training device: GPU (CUDA)
Epoch 1/10, Loss: 37.6306
Epoch 2/10, Loss: 35.1077
Epoch 3/10, Loss: 33.5602
Epoch 4/10, Loss: 31.0902
Epoch 5/10, Loss: 29.8800
Epoch 6/10, Loss: 29.4263
Epoch 7/10, Loss: 27.7595
Epoch 8/10, Loss: 27.0227
Epoch 9/10, Loss: 26.6356
Epoch 10/10, Loss: 25.4584
Window size: 2

----------------------------------------

----------------------------------------

Training device: GPU (CUDA)
Epoch 1/10, Loss: 44.3906
Epoch 2/10, Loss: 41.2358
Epoch 3/10, Loss: 38.6885
Epoch 4/10, Loss: 36.4417
Epoch 5/10, Loss: 34.5508
Epoch 6/10, Loss: 32.8374
Epoch 7/10, Loss: 31.3907
Epoch 8/10, Loss: 30.1095
Epoch 9/10, Loss: 28.9736
Ep

,he,mentioned,reading,a,is,she,bank,close,manager,containing,...,actor,window,exposure,file,lost,report,performance,book,fascinating,harmful
1-2,"[-1.9887941, 0.05634008, -0.6048523, 0.9478458...","[0.53824806, 0.3866042, 0.6949526, 2.6085806, ...","[-0.40769178, 0.33527732, 0.5953762, 0.6546778...","[-0.012275875, -1.8215741, 1.4254977, 0.917209...","[1.9395834, 0.13259572, -0.57696486, 0.5900499...","[0.30795652, -2.2710829, -3.0131059, -0.771548...","[0.028149188, -2.6935403, 0.7799363, 0.5633403...","[-1.1969995, 3.2115068, -1.4111221, 0.78891087...","[1.5561863, -1.358942, 0.60532856, -2.646164, ...","[2.8328438, 0.10387489, 0.91233003, 0.46722552...",...,"[-1.1027701, 1.4688529, -0.56695986, -0.018035...","[0.38628897, -0.8542701, -1.6987693, -2.182338...","[-0.34890997, 2.2575588, 1.2616171, -2.119136,...","[0.9806574, 0.29471588, -1.3251383, -1.0868694...","[-1.754617, -2.3000448, 0.348589, -1.7416439, ...","[-1.470262, 0.48173773, -1.2038456, -0.0611770...","[1.9243042, -0.5912593, -1.6508632, 0.23279738...","[-1.2307936, -0.41228887, 1.6251383, -0.872160...","[0.6579324, -0.521741, 0.31769273, -0.61521614...","[1.4362918, 0.62995344, 0.085748136, 2.6446872..."
2-3,"[-1.5548439, -0.7270155, 1.3866831, 0.4543265,...","[2.7308455, -2.9508085, 2.4818664, -1.2127862,...","[-2.030962, -0.28336066, 1.1402925, -0.3654250...","[0.06799668, -0.43963188, -1.0938339, -0.68390...","[-2.8612363, -1.8988869, 1.4045888, 0.04738113...","[-1.2510135, -0.4478365, 3.840825, 0.5055088, ...","[1.0542047, 2.360484, 2.6976383, -0.527737, -1...","[1.327168, -0.9163699, 2.1125793, 0.69482964, ...","[-1.156617, 1.4578474, -0.4009921, 0.32988298,...","[-1.9451293, 1.1977485, -0.70145595, -1.802537...",...,"[0.062071204, 1.0499122, 0.7970518, 1.6767907,...","[-1.1400257, -0.6114969, 0.6238934, -0.5041037...","[0.5447209, -0.24895698, -0.7974873, 1.2550073...","[-1.3301051, -1.0412381, -0.28793794, 2.741037...","[1.7223591, 2.5496564, 1.7864592, 0.6192951, 2...","[-1.3815408, -1.4546266, 1.4335349, 0.7280985,...","[-1.5380504, -0.58882105, 2.1975102, -0.142253...","[0.0022025108, -0.1978897, -1.2932062, 0.72434...","[0.057875007, 1.5278699, 0.6320256, -0.0982108...","[-0.8805883, -0.07300657, 0.8414837, -1.547730..."
3-4,"[1.3730272, -0.7206221, -0.39172903, -1.392159...","[-1.7431313, 2.5821748, -2.617081, -0.5549159,...","[-0.40854514, -1.3050628, -1.6341053, 0.690308...","[-0.26440218, -0.51745254, 0.4592235, 0.491453...","[0.25831944, 0.21713674, 0.34460098, 0.3645423...","[1.0375974, 0.38512397, -0.5307901, 0.8415423,...","[0.3098484, -0.16378815, -1.733892, -1.6237755...","[-0.57183343, 1.0412953, -3.1465342, 0.3104568...","[-0.15426442, 0.32813272, 1.2879875, 1.616183,...","[1.5409498, -0.3873352, -0.86687994, 1.4191875...",...,"[0.13380682, -0.9672286, 0.08615476, -1.901641...","[0.7185611, -0.13178134, 1.8065143, 1.3791711,...","[0.10391046, 0.0047083497, 0.9049493, -1.41629...","[1.9246184, 0.42778248, 0.8140464, -1.1825135,...","[-1.8394147, -1.0004532, -1.6894238, 0.7141876...","[2.0355804, 2.6956065, 0.820379, -0.9510555, -...","[0.5356042, 0.004749447, -2.0502546, 0.0315784...","[1.524152, -0.9128878, 0.10728639, -0.16749537...","[-1.1883807, 0.25111842, -0.18385497, 0.929130...","[1.4544834, -1.3309456, -0.20262432, -0.213409..."
4-5,"[-1.0910442, 1.7283032, 0.069681965, 0.2931200...","[-1.2214366, -0.6572046, -0.12680012, -0.65830...","[1.3734004, 1.1636326, 1.5220009, -1.7423075, ...","[-0.9486148, 1.0883999, 2.0817437, 0.18417013,...","[0.34215802, 0.6231598, -1.0247389, 1.1046908,...","[0.69482577, 0.12008689, 1.3910004, -1.4749951...","[-1.4482887, -0.673388, -0.063097954, 1.124555...","[-0.7853627, -0.26739776, 1.9668787, -0.704195...","[0.492491, -0.29253447, -0.7677386, -1.3611407...","[-0.19644189, -1.6388828, 1.9343612, 0.3501805...",...,"[0.11618841, -1.2202839, -0.9678398, 0.7194094...","[0.17832516, -0.7557256, -1.162793, -0.3339737...","[0.84209377, 0.21607238, -0.27533633, -0.28960...","[-0.18040472, -0.7386043, -1.2

We can see that there differences between different window sizes. However, it doesn't seem to have a constant trend. Increasing the window size is supposed to make the model be better at recognizing context, and thus the weights should change drastically for double-meaning words (such as bank) and we can see that it does.

### CBOW

## g. Compare these two models 
In terms of capturing the syntactic and the semantic relationship between words.

## h. Demonstrate the difference between CBOW and Skip-grams
In terms of cosine similarity between the following words:
 - bank, rose, lead, book and file.

### Skip-gram

In [17]:
import pandas as pd
from scipy.spatial.distance import cosine

selected_words = ['bank', 'rose', 'lead', 'book', 'file']  # Words to calculate cosine similarity for

# Initialize an empty DataFrame to store cosine similarities
cosine_similarity_df = pd.DataFrame(index=selected_words, columns=selected_words)

# Calculate cosine similarity for selected word pairs
for word1 in selected_words:
    for word2 in selected_words:
        if word1 != word2:
            # Calculate cosine similarity (1 - cosine distance)
            similarity = 1 - cosine(embedding_dict_skipgram[word1], embedding_dict_skipgram[word2])
            cosine_similarity_df.at[word1, word2] = similarity
        else:
            # Similarity with itself is 1
            cosine_similarity_df.at[word1, word2] = 1.0

# Convert to float type (optional)
cosine_similarity_df = cosine_similarity_df.astype(float)

# Display the resulting DataFrame
cosine_similarity_df

,bank,rose,lead,book,file
bank,1.000000,0.589130,-0.137048,0.333292,0.225097
rose,0.589130,1.000000,0.185908,-0.119245,0.256498
lead,-0.137048,0.185908,1.000000,-0.070307,0.232382
book,0.333292,-0.119245,-0.070307,1.000000,0.272276
file,0.225097,0.256498,0.232382,0.272276,1.000000


### CBOW

## i. How can the subword embeddings be applied?
Michael

# 2. Create example sentences demonstrating: 
how contextual embeddings handle words with multiple meanings (polysemy) differently than static embeddings like Word2Vec.
Barak

In [22]:
import tensorflow_hub as hub
import tensorflow as tf
import numpy as np
import nltk
from nltk.tokenize import word_tokenize

# Make sure to download NLTK data (if not already downloaded)
nltk.download('punkt')

# Load ELMo model from TensorFlow Hub
elmo = hub.load("https://tfhub.dev/google/elmo/3")

# Example sentences for demonstrating polysemy
example_sentences = [
    "He went to the bank to deposit some money.",  # bank as a financial institution
    "The fisherman docked his boat near the river bank.",  # bank as a riverbank
    "She rose from her chair to make an announcement.",  # rose as an action/past tense of rise
    "The rose smelled fragrant and bloomed beautifully.",  # rose as a flower
    "The project manager will lead the team during the meeting.",  # lead as to guide
    "Exposure to lead in paint can cause health issues."  # lead as a toxic substance
]

words_to_compare = ["bank", "rose", "lead"]  # Words to compare

# Function to get contextual embeddings using ELMo
def get_elmo_embeddings(sentences, word):
    """
    Extract contextual embeddings for a specific word from ELMo.
    Arguments:
    - sentences: List of sentences to process.
    - word: The target word to extract embeddings for.

    Returns:
    - A dictionary where keys are sentences and values are embeddings of the target word.
    """
    embeddings = {}

    for sentence in sentences:
        elmo_embeddings = elmo.signatures["default"](tf.constant([sentence]))["elmo"]  # Shape: (1, sentence_length, 1024)
        elmo_embeddings_np = elmo_embeddings.numpy().squeeze()  # Shape: (sentence_length, 1024)
        tokens = word_tokenize(sentence)  # Tokenize using nltk
        if word in tokens:
            word_index = tokens.index(word)  # Find the index of the word
            embeddings[sentence] = elmo_embeddings_np[word_index]  # Extract the embedding for the word
        else:
            embeddings[sentence] = None  # Word not found in the sentence
    return embeddings

# Get static embeddings from the Skip-gram Word2Vec model
def get_skipgram_embedding(word, embedding_dict_skipgram):
    """
    Extract the static embedding for a word from the Skip-gram Word2Vec model.
    """
    return embedding_dict_skipgram.get(word)

# Compare Skip-gram and ELMo embeddings
for word in words_to_compare:
    print(f"\nWord: '{word}'")

    # Contextual embeddings with ELMo
    print("Contextual Embeddings (varies by sentence):")
    elmo_embeddings = get_elmo_embeddings(example_sentences, word)
    for sentence, embedding in elmo_embeddings.items():
        if embedding is not None:
            print(f"  Sentence: {sentence}")
            print(f"    Embedding: {embedding[:5]}...")  # Print only the first 5 dimensions for brevity
        else:
            print(f"  Sentence: {sentence}")
            print("    Embedding: Word not found in the sentence.")

    # Static embedding using Skip-gram Word2Vec
    print("Static Embedding (same for all sentences):")
    skipgram_embedding = get_skipgram_embedding(word, embedding_dict_skipgram)
    if skipgram_embedding is not None:
        print(f"  Embedding: {skipgram_embedding[:5]}...")  # Print only the first 5 dimensions for brevity
    else:
        print("  Embedding: Word not found in the Word2Vec vocabulary.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\barak\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!



Word: 'bank'
Contextual Embeddings (varies by sentence):
  Sentence: He went to the bank to deposit some money.
    Embedding: [-0.40484837  0.18932024  0.08644994  0.2684955   0.35230607]...
  Sentence: The fisherman docked his boat near the river bank.
    Embedding: [ 0.10449297  0.3106082  -0.563089   -0.53706545 -0.81824535]...
  Sentence: She rose from her chair to make an announcement.
    Embedding: Word not found in the sentence.
  Sentence: The rose smelled fragrant and bloomed beautifully.
    Embedding: Word not found in the sentence.
  Sentence: The project manager will lead the team during the meeting.
    Embedding: Word not found in the sentence.
  Sentence: Exposure to lead in paint can cause health issues.
    Embedding: Word not found in the sentence.
Static Embedding (same for all sentences):
  Embedding: [ 0.4233067  -0.02789077 -0.7929738   1.0265157  -0.62915695]...

Word: 'rose'
Contextual Embeddings (varies by sentence):
  Sentence: He went to the bank to depo

# 3. Propose metrics
For evaluating word embeddings that can differentiate between syntactic and semantic relationships.
Michael

# 4. Use the Gensim library 
To train a Word2Vec model on a custom corpus.  
Barak

## a. Evaluate the quality of embeddings 
By calculating the cosine similarity for the following word pairs:
```
i. "king" and "queen"  
ii. "man" and "woman"
iii. "apple" and "orange"
```

## b. Write a brief explanation of the results.

# 5. Train the GloVe model 
Using the glove-python package on a subset of a publicly available dataset (e.g., Wikipedia, or a smaller custom corpus).

Michael

## a. Use t-SNE or PCA 
To visualize the embeddings in 2D.

## b. Analyze the clustering patterns observed in the visualization.